### 랭그래프 사칙 연산 도구

In [30]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# 1. 모델 초기화
model = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",
    temperature=0
)

In [31]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """
    두 숫자를 더해 그 합을 반환합니다.

    Args:
        a: 더할 첫 번째 숫자
        b: 더할 두 번째 숫자
    """
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """
    첫 번째 숫자에서 두 번째 숫자를 뺍니다.

    Args:
        a: 빼기 연산의 대상이 되는 수 (피감수)
        b: 뺄 숫자 (감수)
    """
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """
    두 숫자를 곱해 그 결과를 반환합니다.

    Args:
        a: 곱할 첫 번째 숫자
        b: 곱할 두 번째 숫자
    """
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """
    첫 번째 숫자를 두 번째 숫자로 나눕니다. 
    0으로 나누려고 하면 에러 메시지를 반환합니다.

    Args:
        a: 나뉘는 수 (피제수)
        b: 나누는 수 (제수)
    """
    if b == 0:
        return "Error: 0으로 나눌 수 없습니다."
    return a / b

# 툴 리스트
tools = [add, subtract, multiply, divide]

In [32]:
# 3. 모델에 도구 바인딩
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

In [33]:
tools_by_name

{'add': StructuredTool(name='add', description='두 숫자를 더해 그 합을 반환합니다.\n\nArgs:\n    a: 더할 첫 번째 숫자\n    b: 더할 두 번째 숫자', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001C21E371A80>),
 'subtract': StructuredTool(name='subtract', description='첫 번째 숫자에서 두 번째 숫자를 뺍니다.\n\nArgs:\n    a: 빼기 연산의 대상이 되는 수 (피감수)\n    b: 뺄 숫자 (감수)', args_schema=<class 'langchain_core.utils.pydantic.subtract'>, func=<function subtract at 0x000001C21E3719E0>),
 'multiply': StructuredTool(name='multiply', description='두 숫자를 곱해 그 결과를 반환합니다.\n\nArgs:\n    a: 곱할 첫 번째 숫자\n    b: 곱할 두 번째 숫자', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000001C21E371760>),
 'divide': StructuredTool(name='divide', description='첫 번째 숫자를 두 번째 숫자로 나눕니다. \n0으로 나누려고 하면 에러 메시지를 반환합니다.\n\nArgs:\n    a: 나뉘는 수 (피제수)\n    b: 나누는 수 (제수)', args_schema=<class 'langchain_core.utils.pydantic.divide'>, func=<function divide at 0x000001C21E373920>)}

In [34]:
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage
import operator

# 4. MessageState 정의
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [35]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from typing import Literal

# 5. Node 정의
def llm_call(state: MessagesState):
    """LLM이 도구를 호출 여부를 결정합니다."""

    system_prompt = (
        "당신은 친절한 도우미입니다.\n"
        "사용자의 질문에 답변하고, 연산은 도구를 사용하세요.\n"
    )

    messages = [
        SystemMessage(content=system_prompt)
    ] + state["messages"]

    response = model_with_tools.invoke(messages)
    return {"messages": [response]}


# 6. ToolNode 정의
def tool_node(state: MessagesState):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}


# 7. 조건 함수 정의
def should_continue(state: MessagesState) -> Literal["tool_node", END]: # type: ignore
    """LLM이 도구 호출 여부에 따라 반복을 계속할지 멈출지를 결정합니다."""

    messages = state["messages"]
    last_message = messages[-1]

    if last_message.tool_calls:
        return "tool_node"
    return END

In [36]:
from langgraph.graph import StateGraph, START, END, add_messages

# 8. Graph 생성
graph_builder = StateGraph(MessagesState)

# 9. Graph에 Node 추가
graph_builder.add_node("llm_call", llm_call)
graph_builder.add_node("tool_node", tool_node)

# 10. Graph에 Edge 추가
graph_builder.add_edge(START, "llm_call")
graph_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
graph_builder.add_edge("tool_node", "llm_call")
graph_builder.add_edge("llm_call", END)

# 11. 컴파일
graph = graph_builder.compile()

In [42]:
# 12. 실행 예시
# human_message = HumanMessage(content="6 더하기 5는?")
human_message = HumanMessage(content="123 + 456 * 789 / 12은 뭔가요?")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

123 + 456 * 789 / 12은 뭔가요?
================================== Ai Message ==================================

계산 순서에 따라 456 * 789를 먼저 계산한 다음 그 결과값을 123에 더하고, 마지막으로 그 결과값을 12로 나누겠습니다. 먼저 456 * 789를 계산하겠습니다.
Tool Calls:
  multiply (f1c385a6-d694-410f-8f0b-12f470fc8204)
 Call ID: f1c385a6-d694-410f-8f0b-12f470fc8204
  Args:
    a: 456
    b: 789
================================= Tool Message =================================

359784
================================== Ai Message ==================================
Tool Calls:
  divide (85772db2-ac6d-47d9-9c56-ef22600c7cf6)
 Call ID: 85772db2-ac6d-47d9-9c56-ef22600c7cf6
  Args:
    a: 359784
    b: 12
================================= Tool Message =================================

29982.0
================================== Ai Message ==================================
Tool Calls:
  add (ce374f07-6224-4497-b493-d826a68c9b24)
 Call ID: ce374f07-6224-4497-b493-d82

In [39]:
result

{'messages': [HumanMessage(content='2 곱하기 3을 한 다음에 5를 더하고 그 결과를 2로 나눠줘.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"a": 6, "b": 5}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c604f-a6d4-7a12-88c4-efb6c3ddcd00-0', tool_calls=[{'name': 'add', 'args': {'a': 6, 'b': 5}, 'id': '613ed548-5af6-4eec-9c29-9893d00474e3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 384, 'output_tokens': 18, 'total_tokens': 402, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='11', tool_call_id='613ed548-5af6-4eec-9c29-9893d00474e3'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c604f-aa47-7a13-a13f-d4ce